# Causal Uplift Modeling: End-to-End Analysis
This notebook walks through the results of our heterogeneous treatment effect (HTE) modeling pipeline on the Criteo dataset.
We trained several meta-learners (S, T, X, R) and a Causal Forest to estimate individual-level treatment effects (CATE).

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

# Load evaluation metrics
with open('../results/metrics.json', 'r') as f:
    metrics = json.load(f)

## 1. Overall Model Performance
We evaluate the models using Area Under the Uplift Curve (AUUC) and the Qini coefficient.

In [ ]:
# Format metrics into a clean DataFrame
records = []
for model_name, data in metrics.items():
    records.append({
        'Model': model_name,
        'AUUC': data['auuc'],
        'Qini': data['qini'],
        'Calibration Error': data['cal_error'],
        'Policy Value @ 5%': data['policy_values']['0.05']
    })

df_metrics = pd.DataFrame(records).sort_values('AUUC', ascending=False)
display(df_metrics.style.highlight_max(subset=['AUUC', 'Qini', 'Policy Value @ 5%'], color='lightgreen'))

## 2. Propensity Score & Overlap
Ensuring sufficient overlap between the treated and control populations is a foundational assumption in causal inference (positivity).

In [ ]:
display(Image(filename='../results/figures/01_propensity_overlap.png'))

## 3. Uplift & Qini Curves
These curves show how much cumulative incremental lift we get by targeting the top X% of users ranked by their predicted CATE.

In [ ]:
display(Image(filename='../results/figures/02_auuc_comparison.png'))

In [ ]:
display(Image(filename='../results/figures/03_qini_curves.png'))

## 4. Policy Values
Expected average treatment effect if we implement a policy targeting only the top fractions.

In [ ]:
display(Image(filename='../results/figures/06_policy_value_curves.png'))

## 5. Model Calibration
How well do the predicted CATEs align with the actual observed Average Treatment Effects (ATE) by decile?

In [ ]:
display(Image(filename='../results/figures/04_cate_calibration.png'))

## 6. Feature Importance & Placebo Test
We run a placebo test by randomly shuffling the treatment assignments and retraining. The resulting AUUC drops near zero, confirming our real models are picking up genuine signal rather than noise.

In [ ]:
display(Image(filename='../results/figures/05_shap_summary.png'))

In [ ]:
display(Image(filename='../results/figures/07_placebo_test.png'))